In [1]:
import glob

import pandas as pd

from datasets import calcular_cortes_split, elegibilidad_pozos, generar_ventanas, process_dataset

df = process_dataset("datos_modelo.csv")

print(f"df: {df.shape}  —  {df['idpozo'].nunique():,} pozos únicos")

print(df.info())


df: (39562, 15)  —  497 pozos únicos
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39562 entries, 0 to 39561
Data columns (total 15 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   idpozo                       39562 non-null  int64  
 1   profundidad_pozo             39562 non-null  float64
 2   longitud_rama_horizontal_m   39562 non-null  float64
 3   cantidad_fracturas           39562 non-null  int64  
 4   arena_bombeada_nacional_tn   39562 non-null  float64
 5   arena_bombeada_importada_tn  39562 non-null  float64
 6   agua_inyectada_m3            39562 non-null  float64
 7   presion_maxima_psi           39562 non-null  float64
 8   anio_prod                    39562 non-null  int64  
 9   mes_prod                     39562 non-null  int64  
 10  prod_pet                     39562 non-null  float64
 11  coordenadax                  39562 non-null  float64
 12  coordenaday                  39562 no

In [2]:
cortes = calcular_cortes_split(df, input_len=36, target_len=12, min_ventanas_train=12)
resumen_pozo = elegibilidad_pozos(df, cortes)

pozos_train = resumen_pozo.index[resumen_pozo["elegible_train"]]
pozos_val = resumen_pozo.index[resumen_pozo["elegible_val"]]
pozos_test = resumen_pozo.index[resumen_pozo["elegible_test"]]

print(f"Pozos elegibles — train: {len(pozos_train):,}  val: {len(pozos_val):,}  test: {len(pozos_test):,}")


Pozos elegibles — train: 205  val: 342  test: 497


In [3]:
ventanas_train = generar_ventanas(df, cortes, "train", pozos_train)
ventanas_val = generar_ventanas(df, cortes, "val", pozos_val)
ventanas_test = generar_ventanas(df, cortes, "test", pozos_test)

print(f"ventanas_train: {ventanas_train.shape[0]:,} filas  —  {ventanas_train['idpozo'].nunique():,} pozos")
print(f"ventanas_val  : {ventanas_val.shape[0]:,} filas  —  {ventanas_val['idpozo'].nunique():,} pozos")
print(f"ventanas_test : {ventanas_test.shape[0]:,} filas  —  {ventanas_test['idpozo'].nunique():,} pozos")

ventanas_por_pozo = ventanas_train.groupby("idpozo").size()
print(f"\nVentanas por pozo en train — min: {ventanas_por_pozo.min()}  max: {ventanas_por_pozo.max()}  mediana: {ventanas_por_pozo.median():.0f}")

print("\nEjemplo — todas las ventanas de un pozo en train:")
idpozo_ejemplo = ventanas_por_pozo.index[0]
display(ventanas_train[ventanas_train["idpozo"] == idpozo_ejemplo])

print("\nEjemplo — primeras filas de ventanas_val y ventanas_test:")
display(ventanas_val.head())
display(ventanas_test.head())


ventanas_train: 7,094 filas  —  205 pozos
ventanas_val  : 342 filas  —  342 pozos
ventanas_test : 497 filas  —  497 pozos

Ventanas por pozo en train — min: 12  max: 74  mediana: 33

Ejemplo — todas las ventanas de un pozo en train:


,idpozo,input_start,target_start
0,147488,2014-05,2017-05
1,147488,2014-06,2017-06
2,147488,2014-07,2017-07
3,147488,2014-08,2017-08
4,147488,2014-09,2017-09
...,...,...,...
69,147488,2020-02,2023-02
70,147488,2020-03,2023-03
71,147488,2020-04,2023-04
72,147488,2020-05,2023-05



Ejemplo — primeras filas de ventanas_val y ventanas_test:


,idpozo,input_start,target_start
0,147488,2021-06,2024-06
1,152993,2021-06,2024-06
2,153620,2021-06,2024-06
3,153622,2021-06,2024-06
4,153773,2021-06,2024-06


,idpozo,input_start,target_start
0,147488,2022-06,2025-06
1,152993,2022-06,2025-06
2,153620,2022-06,2025-06
3,153622,2022-06,2025-06
4,153773,2022-06,2025-06


In [4]:
print("edad_pozo — estadísticas generales:")
print(df["edad_pozo"].describe())

edad_min_por_pozo = df.groupby("idpozo")["edad_pozo"].min()
print(f"\nPozos con edad_pozo mínima != 0: {(edad_min_por_pozo != 0).sum()}  (debería ser 0)")

print("\nEjemplo — evolución de edad_pozo para un pozo:")
idpozo_ejemplo = df["idpozo"].iloc[0]
display(
    df[df["idpozo"] == idpozo_ejemplo]
    .sort_values(["anio_prod", "mes_prod"])
    [["idpozo", "anio_prod", "mes_prod", "edad_pozo"]]
    .head(10)
)


edad_pozo — estadísticas generales:
count    39562.000000
mean        43.272686
std         29.365157
min          0.000000
25%         19.000000
50%         39.000000
75%         62.000000
max        144.000000
Name: edad_pozo, dtype: float64

Pozos con edad_pozo mínima != 0: 0  (debería ser 0)

Ejemplo — evolución de edad_pozo para un pozo:


,idpozo,anio_prod,mes_prod,edad_pozo
58,147488,2014,5,0
69,147488,2014,6,1
77,147488,2014,7,2
95,147488,2014,8,3
103,147488,2014,9,4
110,147488,2014,10,5
125,147488,2014,11,6
142,147488,2014,12,7
8,147488,2015,1,8
22,147488,2015,2,9


In [5]:
from datasets import calcular_normalizacion

normalizacion = calcular_normalizacion(df, cortes, pozos_train)

pozos_totales = df["idpozo"].nunique()
pozos_fuera_de_train = pozos_totales - len(pozos_train)

print(f"normalizacion: {normalizacion.shape}  —  {normalizacion.index.nunique():,} pozos")
print(f"Pozos con fallback global : {normalizacion['fallback_global'].sum():,}")
print(f"Pozos fuera de pozos_train: {pozos_fuera_de_train:,}  (debería coincidir con la línea anterior)")

print("\nEjemplo — pozos CON historia en train (estadístico propio, distinto por pozo):")
display(normalizacion.loc[normalizacion.index.isin(pozos_train)].head(3))

print("\nEjemplo — pozos SIN historia en train (fallback global, mismo valor para todos):")
display(normalizacion.loc[~normalizacion.index.isin(pozos_train)].head(3))


normalizacion: (497, 3)  —  497 pozos
Pozos con fallback global : 292
Pozos fuera de pozos_train: 292  (debería coincidir con la línea anterior)

Ejemplo — pozos CON historia en train (estadístico propio, distinto por pozo):


,media_log,std_log,fallback_global
idpozo,,,
147488,4.030412,2.799391,False
152993,5.146650,1.656675,False
153620,5.223455,2.279845,False



Ejemplo — pozos SIN historia en train (fallback global, mismo valor para todos):


,media_log,std_log,fallback_global
idpozo,,,
160885,5.741957,2.089435,True
160886,5.741957,2.089435,True
160897,5.741957,2.089435,True


In [6]:
from datasets import calcular_normalizacion_regresores

normalizacion_regresores = calcular_normalizacion_regresores(df, cortes, pozos_train)
print("normalizacion_regresores (fit sobre pozos_train):")
display(normalizacion_regresores)

# ── Verificación de que el fit usa SOLO pozos_train (no leakage) ───────────
# Comparo contra el mismo cálculo usando TODOS los pozos: si el filtro de
# pozos_train está actuando, los resultados tienen que diferir.
normalizacion_regresores_todos = calcular_normalizacion_regresores(df, cortes, df["idpozo"].unique())

diff = (normalizacion_regresores["media"] - normalizacion_regresores_todos["media"]).abs()
print("\nDiferencia de 'media' (fit con pozos_train vs. fit con TODOS los pozos):")
print(diff.to_string())
print(f"\nTodas las columnas difieren: {(diff > 1e-9).all()}  (confirma que el filtro de pozos_train realmente actúa)")


normalizacion_regresores (fit sobre pozos_train):


,media,std,log1p
columna,,,
arena_bombeada_importada_tn,5.552912,2.645833,True
profundidad_pozo,4991.731415,580.491251,False
longitud_rama_horizontal_m,1748.464341,494.187435,False
cantidad_fracturas,24.112195,9.869844,False
arena_bombeada_nacional_tn,4284.151978,2210.010895,False
agua_inyectada_m3,30057.109416,14173.591059,False
presion_maxima_psi,11425.337827,783.700484,False
coordenadax,-68.665289,0.104176,False
coordenaday,-38.318925,0.108212,False



Diferencia de 'media' (fit con pozos_train vs. fit con TODOS los pozos):
columna
arena_bombeada_importada_tn        2.145139
profundidad_pozo                 371.626652
longitud_rama_horizontal_m       405.590789
cantidad_fracturas                 8.475330
arena_bombeada_nacional_tn      2444.374561
agua_inyectada_m3              16565.321459
presion_maxima_psi               335.400709
coordenadax                        0.000632
coordenaday                        0.037440
edad_pozo                          8.680705

Todas las columnas difieren: True  (confirma que el filtro de pozos_train realmente actúa)


In [ ]:
import numpy as np

from datasets import PozoWindowDataset

ds_train = PozoWindowDataset(df, ventanas_train, normalizacion, normalizacion_regresores)
print(f"len(ds_train) = {len(ds_train):,}  (debería ser igual a ventanas_train.shape[0] = {ventanas_train.shape[0]:,})")

ejemplo = ds_train[0]
print("\nClaves del dict devuelto:", list(ejemplo.keys()))
print("Shapes esperadas -> x_prod_pet:(36,1)  x_regressors:(36,11)  y_regressors:(12,11)  y_prod_pet:(12,1)")
for clave, valor in ejemplo.items():
    if hasattr(valor, "shape"):
        print(f"  {clave:15s} shape={tuple(valor.shape)}  dtype={valor.dtype}")
    else:
        print(f"  {clave:15s} valor={valor}")

In [ ]:
from datasets import normalizar_prod_pet

periodo_df = pd.PeriodIndex.from_fields(year=df["anio_prod"], month=df["mes_prod"], freq="M")
serie_pozo = df.assign(periodo=periodo_df).set_index(["idpozo", "periodo"])["prod_pet"]

# ── Chequeo 1: pozo de TRAIN (estadístico propio) ──────────────────────────
idx_check = 0
fila_check = ventanas_train.iloc[idx_check]
idpozo_check = fila_check["idpozo"]

periodos_input = pd.period_range(fila_check["input_start"], periods=36, freq="M")
prod_pet_input = serie_pozo.loc[idpozo_check].loc[periodos_input].to_numpy()

media_log, std_log = normalizacion.loc[idpozo_check, ["media_log", "std_log"]]
x_prod_pet_manual = normalizar_prod_pet(prod_pet_input, media_log, std_log)
x_prod_pet_dataset = ds_train[idx_check]["x_prod_pet"].numpy().ravel()

print(f"Pozo {idpozo_check} (train, estadístico propio):")
print(f"  máxima diferencia absoluta (manual vs. Dataset): {np.abs(x_prod_pet_manual - x_prod_pet_dataset).max():.2e}")

# ── Chequeo 2: pozo de VAL que nunca aparece en train (fallback global) ────
ds_val = PozoWindowDataset(df, ventanas_val, normalizacion, normalizacion_regresores)

pozos_val_unseen = [p for p in ventanas_val["idpozo"] if p not in set(pozos_train)]
idpozo_unseen = pozos_val_unseen[0]
idx_unseen = ventanas_val.index[ventanas_val["idpozo"] == idpozo_unseen][0]

fila_unseen = ventanas_val.loc[idx_unseen]
periodos_input_unseen = pd.period_range(fila_unseen["input_start"], periods=36, freq="M")
prod_pet_input_unseen = serie_pozo.loc[idpozo_unseen].loc[periodos_input_unseen].to_numpy()

media_log_u, std_log_u = normalizacion.loc[idpozo_unseen, ["media_log", "std_log"]]
x_prod_pet_manual_u = normalizar_prod_pet(prod_pet_input_unseen, media_log_u, std_log_u)
x_prod_pet_dataset_u = ds_val[idx_unseen]["x_prod_pet"].numpy().ravel()

print(f"\nPozo {idpozo_unseen} (val, fallback_global={normalizacion.loc[idpozo_unseen, 'fallback_global']}):")
print(f"  máxima diferencia absoluta (manual vs. Dataset): {np.abs(x_prod_pet_manual_u - x_prod_pet_dataset_u).max():.2e}")

In [9]:
from datasets import PozoWindowDataset

# ── Verificación del flag `activo` (x_regressors) y el placeholder (y_regressors) ──

print("Distribución de tipoestado_prod (todo el dataset):")
print(df["tipoestado_prod"].value_counts())

COL_ACTIVO = -1  # última columna de x_regressors/y_regressors

# Busco una ventana de train con al menos un mes inactivo en el input, para
# que la verificación sea sobre un caso no trivial (no todo-activo).
serie_estado = df.assign(periodo=periodo_df).set_index(["idpozo", "periodo"])["tipoestado_prod"]

idx_con_inactivo = None
for i in range(len(ventanas_train)):
    fila = ventanas_train.iloc[i]
    periodos = pd.period_range(fila["input_start"], periods=36, freq="M")
    estados = serie_estado.loc[fila["idpozo"]].loc[periodos]
    if (~estados.isin(PozoWindowDataset.VALORES_TIPOESTADO_ACTIVO)).any():
        idx_con_inactivo = i
        break

print(f"\nVentana de ejemplo con algún mes inactivo: índice {idx_con_inactivo}")

fila = ventanas_train.iloc[idx_con_inactivo]
periodos = pd.period_range(fila["input_start"], periods=36, freq="M")
estados = serie_estado.loc[fila["idpozo"]].loc[periodos]
activo_manual = estados.isin(PozoWindowDataset.VALORES_TIPOESTADO_ACTIVO).to_numpy(dtype="float32")

activo_dataset = ds_train[idx_con_inactivo]["x_regressors"].numpy()[:, COL_ACTIVO]

print(f"Pozo {fila['idpozo']} — meses inactivos en la ventana de input: {(activo_manual == 0).sum()} de {len(activo_manual)}")
print(f"máxima diferencia absoluta (manual vs. Dataset): {np.abs(activo_manual - activo_dataset).max():.2e}")

# y_regressors debe llevar el placeholder constante, NUNCA el estado real del horizonte.
y_regressors_activo = ds_train[idx_con_inactivo]["y_regressors"].numpy()[:, COL_ACTIVO]
print(f"\ny_regressors (columna activo) — valores únicos: {np.unique(y_regressors_activo)}  (debería ser solo [{PozoWindowDataset.VALOR_ACTIVO_PLACEHOLDER}])")

# Metadata de diagnóstico (nunca input del modelo) — estado real del horizonte.
print(f"\ntipoestado_target (metadata): {ds_train[idx_con_inactivo]['tipoestado_target']}")


Distribución de tipoestado_prod (todo el dataset):
tipoestado_prod
Extracción Efectiva         36779
Parado Transitoriamente      2454
Otras Situación Inactivo       80
Otras Situación Activo         71
Abandonado                     28
En Espera de Reparación        15
En Estudio                     10
En Reserva de Gas               2
Name: count, dtype: int64

Ventana de ejemplo con algún mes inactivo: índice 0
Pozo 147488 — meses inactivos en la ventana de input: 2 de 36
máxima diferencia absoluta (manual vs. Dataset): 0.00e+00

y_regressors (columna activo) — valores únicos: [1.]  (debería ser solo [1.0])

tipoestado_target (metadata): ['Extracción Efectiva', 'Extracción Efectiva', 'Extracción Efectiva', 'Extracción Efectiva', 'Extracción Efectiva', 'Extracción Efectiva', 'Extracción Efectiva', 'Extracción Efectiva', 'Extracción Efectiva', 'Extracción Efectiva', 'Extracción Efectiva', 'Extracción Efectiva']


In [10]:
# ── ¿"Otras Situación Activo" se comporta como activo o como inactivo? ──────
# Comparo prod_pet de esa categoría contra un estado claramente activo
# ("Extracción Efectiva") y uno claramente inactivo ("Parado Transitoriamente").

for estado in ["Extracción Efectiva", "Otras Situación Activo", "Parado Transitoriamente"]:
    prod_pet_estado = df.loc[df["tipoestado_prod"] == estado, "prod_pet"]
    pct_en_cero = (prod_pet_estado == 0).mean() * 100
    print(f"{estado!r:30s}  n={len(prod_pet_estado):>6,}  %prod_pet==0={pct_en_cero:5.1f}%")
    print(prod_pet_estado.describe().to_string())
    print()


'Extracción Efectiva'           n=36,779  %prod_pet==0=  0.4%
count    36779.000000
mean      1184.386270
std       1302.257968
min          0.000000
25%        373.500000
50%        744.790000
75%       1519.381500
max      16513.300000

'Otras Situación Activo'        n=    71  %prod_pet==0= 18.3%
count      71.000000
mean     1897.061394
std      1326.888378
min         0.000000
25%      1008.795000
50%      2047.520000
75%      2767.145000
max      4491.750000

'Parado Transitoriamente'       n= 2,454  %prod_pet==0= 95.3%
count    2454.000000
mean       24.620909
std       212.784185
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max      7727.100000



### Verificación: normalización de regresores estáticos y `edad_pozo`

`PozoWindowDataset.__init__` normaliza los 9 regresores estáticos y `edad_pozo`
**una sola vez**, con el fit poblacional de `calcular_normalizacion_regresores`
(a diferencia de `prod_pet`, que se normaliza por pozo, en cada `__getitem__`).
Como ese cálculo no vuelve a ejecutarse en `__getitem__`, acá se verifica que
el resultado sea el correcto:

1. Se recalcula "a mano" (con `normalizar_regresor` + los parámetros de
   `normalizacion_regresores`) el valor normalizado de un regresor estático
   (`longitud_rama_horizontal_m`) y de `edad_pozo`, para una ventana puntual,
   y se compara contra lo que efectivamente devuelve `ds_train` — la
   diferencia debería ser ~0.
2. Como sanity check agregado, se calcula media/std de cada regresor
   estático sobre el primer paso de **todas** las ventanas de train — deberían
   rondar 0/1, porque así se definió el fit (salvo el sesgo esperable por
   pozos con más ventanas, que se repiten más veces en este promedio).


In [11]:
from datasets import normalizar_regresor

# ── Verificación: regresores estáticos y edad_pozo ya vienen normalizados ──

col_ejemplo = "longitud_rama_horizontal_m"
idx_col = ds_train.columnas_regresores_estaticos.index(col_ejemplo)

fila = ventanas_train.iloc[idx_check]
valor_crudo = df.loc[df["idpozo"] == fila["idpozo"], col_ejemplo].iloc[0]
media, std, log1p = normalizacion_regresores.loc[col_ejemplo, ["media", "std", "log1p"]]
valor_manual = normalizar_regresor(valor_crudo, media, std, bool(log1p))

valor_dataset = ds_train[idx_check]["x_regressors"].numpy()[0, idx_col]
print(f"{col_ejemplo} — pozo {fila['idpozo']}:")
print(f"  crudo={valor_crudo}  manual normalizado={valor_manual:.4f}  Dataset={valor_dataset:.4f}")
print(f"  diferencia: {abs(valor_manual - valor_dataset):.2e}")

# edad_pozo (primer mes de la ventana de input)
media_edad, std_edad, _ = normalizacion_regresores.loc["edad_pozo", ["media", "std", "log1p"]]
periodo_inicio = fila["input_start"]
edad_cruda = df.loc[(df["idpozo"] == fila["idpozo"]) & (periodo_df == periodo_inicio), "edad_pozo"].iloc[0]
edad_manual = normalizar_regresor(edad_cruda, media_edad, std_edad, log1p=False)
edad_dataset = ds_train[idx_check]["x_regressors"].numpy()[0, -2]  # penúltima columna: edad_pozo

print(f"\nedad_pozo — pozo {fila['idpozo']}, primer mes de la ventana:")
print(f"  crudo={edad_cruda}  manual normalizado={edad_manual:.4f}  Dataset={edad_dataset:.4f}")
print(f"  diferencia: {abs(edad_manual - edad_dataset):.2e}")

# Sanity check adicional: sobre el primer paso de CADA ventana de train, media/std
# de cada regresor estático debería rondar 0/1 (por definición del fit poblacional).
x_regressors_todos = np.stack([ds_train[i]["x_regressors"].numpy()[0] for i in range(len(ds_train))])
print("\nMedia/std de x_regressors (primer paso de cada ventana, todo train):")
for i, col in enumerate(ds_train.columnas_regresores_estaticos + ["edad_pozo", "activo"]):
    print(f"  {col:30s} media={x_regressors_todos[:, i].mean():6.3f}  std={x_regressors_todos[:, i].std():6.3f}")


longitud_rama_horizontal_m — pozo 147488:
  crudo=1197.5  manual normalizado=-1.1149  Dataset=-1.1149
  diferencia: 3.43e-08

edad_pozo — pozo 147488, primer mes de la ventana:
  crudo=0  manual normalizado=-1.6114  Dataset=-1.6114
  diferencia: 1.97e-08

Media/std de x_regressors (primer paso de cada ventana, todo train):
  profundidad_pozo               media=-0.234  std= 0.926
  longitud_rama_horizontal_m     media=-0.315  std= 0.929
  cantidad_fracturas             media=-0.248  std= 0.947
  arena_bombeada_nacional_tn     media=-0.356  std= 0.933
  arena_bombeada_importada_tn    media= 0.190  std= 0.855
  agua_inyectada_m3              media=-0.330  std= 0.863
  presion_maxima_psi             media=-0.254  std= 0.938
  coordenadax                    media= 0.047  std= 0.912
  coordenaday                    media=-0.043  std= 0.899
  edad_pozo                      media=-0.837  std= 0.561
  activo                         media= 0.967  std= 0.177


### `PozoDataModule` — simulación de lo que hace el Trainer de Lightning

Se instancia el `DataModule` y se recorren sus 5 dataloaders (`train`,
`val_seen`, `val_unseen`, `test_seen`, `test_unseen`) igual que lo haría
`Trainer.fit`/`.validate`/`.test` internamente: se itera **todo** el
dataloader para contar cuántos registros trae cada split, pero solo se
imprime el detalle (shapes + valores) de los primeros 2 batches de cada uno,
para no inundar la salida. `batch_size=2` en esta celda es a propósito, solo
para que la impresión de tensores sea legible.


In [ ]:
import torch

from datasets import PozoDataModule

# precision=3 (menos ruido decimal), sci_mode=False (nada de notación
# científica), linewidth ancho para que corte menos líneas.
torch.set_printoptions(precision=3, sci_mode=False, linewidth=120)

dm = PozoDataModule(
    ruta_datos="datos_modelo.csv",
    batch_size=2,
    input_len=36,
    target_len=12,
    min_ventanas_train=12,
)
dm.setup()

print("Pozos vistos (aportaron a train):", len(set(dm.ds_train.ventanas["idpozo"])))
print(f"len(ds_train)      = {len(dm.ds_train):,}")
print(f"len(ds_val_seen)   = {len(dm.ds_val_seen):,}")
print(f"len(ds_val_unseen) = {len(dm.ds_val_unseen):,}")
print(f"len(ds_test_seen)  = {len(dm.ds_test_seen):,}")
print(f"len(ds_test_unseen)= {len(dm.ds_test_unseen):,}")


def recorrer_split(nombre_split, dataloader):
    """
    Imita lo que hace el Trainer de Lightning con un dataloader: itera TODOS
    los batches (para contar registros totales), pero solo imprime el
    detalle de los primeros 2 — shape del batch completo + el print estándar
    de PyTorch (con corchetes) del primer elemento de cada batch.
    """
    print("=" * 78)
    print(f"SPLIT: {nombre_split}")
    print("=" * 78)

    total_registros = 0
    for i, batch in enumerate(dataloader):
        tamanio_batch = batch["x_prod_pet"].shape[0]
        total_registros += tamanio_batch

        if i < 1:
            print(f"\n--- Batch {i} (tamaño={tamanio_batch}) ---")
            for clave in ["x_prod_pet", "x_regressors", "y_regressors", "y_prod_pet"]:
                tensor = batch[clave]
                print(f"\n{clave}  shape={tuple(tensor.shape)}  (primer elemento del batch)")
                print(tensor[0])
            print(f"\nidpozo: {batch['idpozo'].tolist()}")

    print(f"\nTotal de registros iterados en '{nombre_split}': {total_registros:,}")
    print()


# val_dataloader()/test_dataloader() son los hooks estándar de Lightning
# (Trainer los llama solo) -> apuntan a pozos VISTOS. Las versiones "_unseen"
# son métodos aparte, para invocar a mano (ver docstring de PozoDataModule).
recorrer_split("train", dm.train_dataloader())
recorrer_split("val_seen", dm.val_dataloader())
recorrer_split("val_unseen", dm.val_dataloader_unseen())
recorrer_split("test_seen", dm.test_dataloader())
recorrer_split("test_unseen", dm.test_dataloader_unseen())

### Verificación: metadata `input_periodos`/`target_periodos`

**Qué se testea (objetivo)**: que `input_periodos`/`target_periodos` — la
metadata nueva que arma `PozoWindowDataset.__getitem__` internamente con
`pd.period_range(fila["input_start"], periods=self.input_len, freq="M")` —
representen correctamente la ventana: que arranquen en el mes correcto, que
tengan la longitud correcta (36/12), y que no haya un salto entre el último
mes del input y el primero del target.

**Cómo se testea**: se compara contra `fila_check`, la fila de
`ventanas_train` correspondiente a esa misma ventana (`idx_check=0`) —ya
calculada en una celda anterior, antes de tocar `PozoWindowDataset`—
verificando que (1) `input_periodos[0]`/`target_periodos[0]` coinciden con
`input_start`/`target_start`, (2) las longitudes son 36/12, y (3) hay
continuidad (`input_periodos[-1] + 1 mes == target_periodos[0]`).

**Por qué es adecuado**: `generar_ventanas` calcula un único valor por
ventana (`input_start`, el punto de partida). `PozoWindowDataset` toma ese
mismo valor y lo **expande** a una secuencia completa de 36/12 meses con
`pd.period_range` — esa expansión es código nuevo que no existía antes de
esta tarea, y es justo lo que hay que probar. No es un cross-check entre dos
implementaciones independientes del mismo cálculo (eso ya lo cubren los
tests sobre `generar_ventanas` en sí, en otra celda) — es una verificación
de que la expansión nueva no corrompe el punto de partida que la Capa 1 ya
había definido como correcto.


In [13]:
# Reconstruyo ds_train por si el kernel no lo tiene fresco (usa las variables
# df/ventanas_train/normalizacion/normalizacion_regresores ya calculadas arriba).
ds_train = PozoWindowDataset(df, ventanas_train, normalizacion, normalizacion_regresores)

idx_check = 0
fila_check = ventanas_train.iloc[idx_check]
ejemplo = ds_train[idx_check]

input_periodos = ejemplo["input_periodos"]
target_periodos = ejemplo["target_periodos"]

print(f"Pozo {fila_check['idpozo']} — ventana índice {idx_check}")
print(f"input_periodos : {len(input_periodos)} valores, primero={input_periodos[0]}, último={input_periodos[-1]}")
print(f"target_periodos: {len(target_periodos)} valores, primero={target_periodos[0]}, último={target_periodos[-1]}")

# ── Cross-check contra generar_ventanas (fuente independiente) ─────────────
print(f"\nfila_check['input_start']  = {fila_check['input_start']}   (debería == input_periodos[0])")
print(f"fila_check['target_start'] = {fila_check['target_start']}   (debería == target_periodos[0])")

assert input_periodos[0] == str(fila_check["input_start"]), "input_periodos[0] no coincide con generar_ventanas"
assert target_periodos[0] == str(fila_check["target_start"]), "target_periodos[0] no coincide con generar_ventanas"
assert len(input_periodos) == 36 and len(target_periodos) == 12, "longitudes inesperadas"

# Continuidad: el mes siguiente al último del input tiene que ser el primero del target.
siguiente_a_input = pd.Period(input_periodos[-1], freq="M") + 1
assert str(siguiente_a_input) == target_periodos[0], "hay un salto entre input y target"

print("\n✓ Todos los chequeos de continuidad pasaron.")

# ── Confirmar que colaciona bien en un DataLoader ───────────────────────────
# DataLoader colaciona listas de Python TRANSPONIENDO: batch["input_periodos"]
# termina siendo una lista de 36 posiciones, cada una una tupla con un valor
# por ejemplo del batch -- por eso accedo por posición externa (índice 0 =
# "mes de arranque de cada ejemplo"), no iterando la lista completa.
from torch.utils.data import DataLoader

loader_prueba = DataLoader(ds_train, batch_size=3, shuffle=False)
batch_prueba = next(iter(loader_prueba))

mes_arranque_por_ejemplo = batch_prueba["input_periodos"][0]
print(f"\nEjemplo de batch (batch_size=3) — mes de arranque (posición 0) de cada uno de los 3 ejemplos:")
print(mes_arranque_por_ejemplo)
print(f"(debería coincidir con ventanas_train['input_start'].head(3): {ventanas_train['input_start'].head(3).tolist()})")


Pozo 147488 — ventana índice 0
input_periodos : 36 valores, primero=2014-05, último=2017-04
target_periodos: 12 valores, primero=2017-05, último=2018-04

fila_check['input_start']  = 2014-05   (debería == input_periodos[0])
fila_check['target_start'] = 2017-05   (debería == target_periodos[0])

✓ Todos los chequeos de continuidad pasaron.

Ejemplo de batch (batch_size=3) — mes de arranque (posición 0) de cada uno de los 3 ejemplos:
('2014-05', '2014-06', '2014-07')
(debería coincidir con ventanas_train['input_start'].head(3): [Period('2014-05', 'M'), Period('2014-06', 'M'), Period('2014-07', 'M')])
